# AgroSele — BERTimbau Congelado + MLP

Segunda variante do projeto: em vez de contar palavra, uso o **BERTimbau**
(`neuralmind/bert-base-portuguese-cased`) pra gerar um embedding de cada
pergunta e cada resposta, e treino um classificador (MLP) por cima pra
decidir se um par (pergunta, candidata) combina ou não.

O BERT fica **totalmente congelado** aqui (nenhum peso dele é atualizado) —
só uso ele como extrator de features. A ideia de fine-tuning (ajustar os
pesos do BERT também) fica pro notebook seguinte.

Arquitetura:

```
pergunta / resposta
      ↓
BERTimbau (congelado) → embedding de 768 dimensões (mean pooling)
      ↓
par (pergunta, candidata):
  vetor = [emb_pergunta, emb_resposta, |diferença|, produto]   (3072d)
      ↓
MLP binário → match / não-match
      ↓
ranking das 50 candidatas pela pontuação
```

Métricas: Accuracy@1 e MRR (é ranqueamento, não classificação de classe fixa).

In [1]:
import itertools
import os
import random
import time

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from datasets import load_dataset

SEMENTE = 42
random.seed(SEMENTE)
np.random.seed(SEMENTE)

C:\Users\frede\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Embeddings do BERTimbau

Extrair embedding de 2657 perguntas + 2657 respostas em CPU demora uns
20-25 minutos (o texto passa pelo BERT em lotes de 16, com mean pooling
sobre a última camada escondida). Já rodei isso antes e salvei em
`cache/*.pt` — então aqui eu só carrego o cache se ele já existir, e só
recalculo do zero se não existir (por exemplo, rodando o projeto pela
primeira vez em outra máquina).

In [2]:
CAMINHO_CACHE_RESPOSTAS = "cache/corpus_embeddings.pt"
CAMINHO_CACHE_PERGUNTAS = "cache/queries_embeddings.pt"
NOME_MODELO = "neuralmind/bert-base-portuguese-cased"
TAMANHO_MAX_TOKENS = 256
TAMANHO_LOTE = 16


def pooling_media(ultima_camada_escondida, mascara_atencao):
    """Faz a media dos vetores de token, ignorando padding (mascara=0)."""
    mascara = mascara_atencao.unsqueeze(-1).expand(ultima_camada_escondida.size()).float()
    soma = torch.sum(ultima_camada_escondida * mascara, dim=1)
    contagem = torch.clamp(mascara.sum(dim=1), min=1e-9)
    return soma / contagem


def gerar_embeddings(textos, tokenizador, modelo, device):
    lista_embeddings = []
    with torch.no_grad():
        for i in range(0, len(textos), TAMANHO_LOTE):
            lote = textos[i:i + TAMANHO_LOTE]
            entrada = tokenizador(lote, return_tensors="pt", truncation=True,
                                   max_length=TAMANHO_MAX_TOKENS, padding=True)
            entrada = {k: v.to(device) for k, v in entrada.items()}
            saida = modelo(**entrada)
            emb = pooling_media(saida.last_hidden_state, entrada["attention_mask"]).cpu()
            lista_embeddings.append(emb)
    return torch.cat(lista_embeddings, dim=0)


if os.path.exists(CAMINHO_CACHE_RESPOSTAS) and os.path.exists(CAMINHO_CACHE_PERGUNTAS):
    print("Cache de embeddings ja existe, carregando direto (nao precisa rodar o BERT de novo)...")
    emb_respostas = torch.load(CAMINHO_CACHE_RESPOSTAS, weights_only=False)
    emb_perguntas = torch.load(CAMINHO_CACHE_PERGUNTAS, weights_only=False)
else:
    print("Cache nao encontrado -- vou extrair os embeddings do zero (demora uns 20-25min em CPU)...")
    from transformers import AutoModel, AutoTokenizer

    os.makedirs("cache", exist_ok=True)
    corpus = load_dataset("eduagarcia/MilkQA", "corpus")["corpus"]
    queries = load_dataset("eduagarcia/MilkQA", "queries")["queries"]

    tokenizador = AutoTokenizer.from_pretrained(NOME_MODELO)
    modelo = AutoModel.from_pretrained(NOME_MODELO)
    modelo.eval()
    for p in modelo.parameters():
        p.requires_grad = False
    device = "cuda" if torch.cuda.is_available() else "cpu"
    modelo.to(device)

    vetores_respostas = gerar_embeddings(corpus["text"], tokenizador, modelo, device)
    emb_respostas = {id_: vetores_respostas[i] for i, id_ in enumerate(corpus["id"])}
    torch.save(emb_respostas, CAMINHO_CACHE_RESPOSTAS)

    vetores_perguntas = gerar_embeddings(queries["text"], tokenizador, modelo, device)
    emb_perguntas = {id_: vetores_perguntas[i] for i, id_ in enumerate(queries["id"])}
    torch.save(emb_perguntas, CAMINHO_CACHE_PERGUNTAS)

print(f"{len(emb_perguntas)} perguntas | {len(emb_respostas)} respostas | dim={next(iter(emb_perguntas.values())).shape[0]}")

Cache de embeddings ja existe, carregando direto (nao precisa rodar o BERT de novo)...
2657 perguntas | 2657 respostas | dim=768


## 1.1 Verificação: o cache é fiel ao BERT de verdade?

Ponto importante de honestidade metodológica: os embeddings acima vieram de
um *cache* em disco, não foram recalculados agora. Como é sempre a mesma
pergunta "isso não seria só copiar um resultado antigo?", a célula abaixo
pega 5 perguntas ao acaso, roda o BERTimbau **ao vivo** só para elas, e
confere se bate com o que está salvo no cache. Se bater (diferença
numérica desprezível), confirma que o cache é fiel — o BERT congelado é
determinístico, então recalcular do zero tem que dar o mesmo vetor.

In [3]:
import random as _random

print("Verificando fidelidade do cache: recalculando 5 exemplos ao vivo...")
from transformers import AutoModel, AutoTokenizer

_ids_amostra = _random.Random(123).sample(list(emb_perguntas.keys()), 5)
_textos_perguntas_csv = {}
with open("datasets/queries.csv", encoding="utf-8") as f:
    import csv as _csv
    for _linha in _csv.DictReader(f):
        _textos_perguntas_csv[_linha["id"]] = _linha["text"]

_tok_verif = AutoTokenizer.from_pretrained(NOME_MODELO)
_modelo_verif = AutoModel.from_pretrained(NOME_MODELO)
_modelo_verif.eval()

_diffs = []
with torch.no_grad():
    for _id in _ids_amostra:
        _entrada = _tok_verif([_textos_perguntas_csv[_id]], return_tensors="pt",
                               truncation=True, max_length=TAMANHO_MAX_TOKENS, padding=True)
        _saida = _modelo_verif(**_entrada)
        _vetor_ao_vivo = pooling_media(_saida.last_hidden_state, _entrada["attention_mask"])[0]
        _vetor_cache = emb_perguntas[_id]
        _diff = (_vetor_ao_vivo - _vetor_cache).abs().max().item()
        _diffs.append(_diff)
        print(f"  pergunta {_id}: diferenca maxima entre cache e recalculo ao vivo = {_diff:.2e}")

assert max(_diffs) < 1e-4, "cache NAO bate com o recalculo ao vivo do BERT!"
print("Cache confirmado: os vetores salvos batem com o recalculo ao vivo do BERTimbau.")
del _modelo_verif, _tok_verif  # libera memoria, nao precisamos mais dele

Verificando fidelidade do cache: recalculando 5 exemplos ao vivo...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 24498.58it/s]


[transformers] BertModel LOAD REPORT from: neuralmind/bert-base-portuguese-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  pergunta 2444: diferenca maxima entre cache e recalculo ao vivo = 2.38e-07
  pergunta 9969: diferenca maxima entre cache e recalculo ao vivo = 2.38e-07
  pergunta 3679: diferenca maxima entre cache e recalculo ao vivo = 2.68e-07
  pergunta 14273: diferenca maxima entre cache e recalculo ao vivo = 2.09e-07


  pergunta 9938: diferenca maxima entre cache e recalculo ao vivo = 2.25e-07
Cache confirmado: os vetores salvos batem com o recalculo ao vivo do BERTimbau.


## 2. Montagem dos pares (pergunta, candidata) → vetor de features

Cada par vira um vetor `[emb_pergunta, emb_resposta, |diferença|, produto]`
(estilo InferSent, Conneau et al. 2017) — 4 blocos de 768d = 3072 dimensões.

Pra montar o treino, uso 1 exemplo positivo (a resposta certa) + 8 negativos
(respostas erradas) por pergunta. Metade dos negativos é escolhida por
**hard negative mining**: pego as candidatas erradas MAIS parecidas com a
pergunta (maior cosseno no embedding cru) em vez de sortear todo mundo à
toa — assim o MLP aprende a distinção fina que realmente importa.

In [4]:
N_NEGATIVOS_TREINO = 8
FRACAO_NEGATIVOS_DIFICEIS = 0.5  # metade dos negativos escolhida por hard negative mining


def features_do_par(emb_pergunta, emb_resposta):
    """[pergunta, resposta, |diferenca|, produto] -- 3072 dimensoes."""
    return torch.cat([
        emb_pergunta,
        emb_resposta,
        torch.abs(emb_pergunta - emb_resposta),
        emb_pergunta * emb_resposta,
    ], dim=-1)


def montar_pares(conjunto, emb_perguntas, emb_respostas, n_negativos, fracao_dificeis=0.0):
    linhas = list(conjunto)
    n_dificeis = int(round(n_negativos * fracao_dificeis))
    n_aleatorios = n_negativos - n_dificeis

    X, y = [], []
    for linha in linhas:
        id_pergunta = linha["query-id"]
        id_resposta_certa = linha["positive-doc-id"]
        candidatas = [c for c in linha["candidates-ids"] if c != id_resposta_certa]

        if n_dificeis > 0 and len(candidatas) > n_dificeis:
            pergunta_norm = torch.nn.functional.normalize(emb_perguntas[id_pergunta].unsqueeze(0), dim=-1)
            candidatas_norm = torch.nn.functional.normalize(
                torch.stack([emb_respostas[c] for c in candidatas]), dim=-1)
            similaridades = (candidatas_norm @ pergunta_norm.T).squeeze(-1)
            indices_dificeis = torch.argsort(-similaridades)[:n_dificeis].tolist()
            negativos_dificeis = [candidatas[i] for i in indices_dificeis]
            restantes = [c for c in candidatas if c not in negativos_dificeis]
            negativos_aleatorios = random.sample(restantes, min(n_aleatorios, len(restantes)))
            negativos = negativos_dificeis + negativos_aleatorios
        else:
            negativos = random.sample(candidatas, min(n_negativos, len(candidatas)))

        emb_p = emb_perguntas[id_pergunta]
        X.append(features_do_par(emb_p, emb_respostas[id_resposta_certa])); y.append(1)
        for id_neg in negativos:
            X.append(features_do_par(emb_p, emb_respostas[id_neg])); y.append(0)

    return torch.stack(X), torch.tensor(y, dtype=torch.float32)

## 3. O modelo (MLP) e a avaliação por ranking

In [5]:
class MLPDoPar(nn.Module):
    def __init__(self, dim_entrada, ocultas, dropout):
        super().__init__()
        self.rede = nn.Sequential(
            nn.Linear(dim_entrada, ocultas), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(ocultas, ocultas // 2), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(ocultas // 2, 1),
        )

    def forward(self, x):
        return self.rede(x).squeeze(-1)


def avaliar_cosseno_puro(conjunto, emb_perguntas, emb_respostas):
    """Baseline sem treino nenhum: so ranqueia pela similaridade de cosseno crua."""
    lista_acuracia1, lista_mrr = [], []
    for linha in conjunto:
        id_pergunta, id_certa, candidatas = linha["query-id"], linha["positive-doc-id"], linha["candidates-ids"]
        pergunta = torch.nn.functional.normalize(emb_perguntas[id_pergunta].unsqueeze(0), dim=-1)
        candidatas_emb = torch.nn.functional.normalize(torch.stack([emb_respostas[c] for c in candidatas]), dim=-1)
        pontuacoes = (candidatas_emb @ pergunta.T).squeeze(-1).numpy()

        ordem = np.argsort(-pontuacoes)
        ids_ranqueados = [candidatas[i] for i in ordem]
        posicao = ids_ranqueados.index(id_certa) + 1

        lista_acuracia1.append(1.0 if posicao == 1 else 0.0)
        lista_mrr.append(1.0 / posicao)
    return float(np.mean(lista_acuracia1)), float(np.mean(lista_mrr))


def avaliar_ranking(modelo, conjunto, emb_perguntas, emb_respostas):
    modelo.eval()
    lista_acuracia1, lista_mrr = [], []
    with torch.no_grad():
        for linha in conjunto:
            id_pergunta, id_certa, candidatas = linha["query-id"], linha["positive-doc-id"], linha["candidates-ids"]
            pergunta = emb_perguntas[id_pergunta].unsqueeze(0).expand(len(candidatas), -1)
            respostas = torch.stack([emb_respostas[c] for c in candidatas])
            features = features_do_par(pergunta, respostas)
            pontuacoes = torch.sigmoid(modelo(features)).numpy()

            ordem = np.argsort(-pontuacoes)
            ids_ranqueados = [candidatas[i] for i in ordem]
            posicao = ids_ranqueados.index(id_certa) + 1

            lista_acuracia1.append(1.0 if posicao == 1 else 0.0)
            lista_mrr.append(1.0 / posicao)
    return float(np.mean(lista_acuracia1)), float(np.mean(lista_mrr))

## 4. Treino com grid search

Testo `hidden_size ∈ {128, 256}`, `dropout ∈ {0.2, 0.4}`, `lr ∈ {1e-3, 1e-4}`
(8 combinações), escolhendo pelo MRR no conjunto de validação oficial. Cada
combinação treina por até 25 épocas, com *early stopping* se o MRR de
validação não melhorar por 4 épocas seguidas.

In [6]:
EPOCAS = 25
PACIENCIA = 4
GRADE_HIPERPARAMETROS = {
    "ocultas": [128, 256],
    "dropout": [0.2, 0.4],
    "taxa_aprendizado": [1e-3, 1e-4],
}


def treinar_um_modelo(X_treino, y_treino, conjunto_dev, emb_perguntas, emb_respostas,
                       ocultas, dropout, taxa_aprendizado):
    torch.manual_seed(SEMENTE)
    modelo = MLPDoPar(X_treino.shape[1], ocultas, dropout)
    otimizador = torch.optim.Adam(modelo.parameters(), lr=taxa_aprendizado)
    funcao_perda = nn.BCEWithLogitsLoss()

    melhor_mrr, melhor_estado, sem_melhora = -1.0, None, 0
    n = X_treino.shape[0]
    tamanho_lote = 64
    gerador = torch.Generator().manual_seed(SEMENTE)

    for epoca in range(EPOCAS):
        modelo.train()
        permutacao = torch.randperm(n, generator=gerador)
        for i in range(0, n, tamanho_lote):
            indices = permutacao[i:i + tamanho_lote]
            otimizador.zero_grad()
            logits = modelo(X_treino[indices])
            perda = funcao_perda(logits, y_treino[indices])
            perda.backward()
            otimizador.step()

        _, mrr_dev = avaliar_ranking(modelo, conjunto_dev, emb_perguntas, emb_respostas)
        if mrr_dev > melhor_mrr:
            melhor_mrr = mrr_dev
            melhor_estado = {k: v.clone() for k, v in modelo.state_dict().items()}
            sem_melhora = 0
        else:
            sem_melhora += 1
            if sem_melhora >= PACIENCIA:
                break

    modelo.load_state_dict(melhor_estado)
    return modelo, melhor_mrr

In [7]:
print("Carregando splits oficiais do MilkQA...")
ds = load_dataset("eduagarcia/MilkQA")
conjunto_treino, conjunto_dev, conjunto_teste = ds["train"], ds["dev"], ds["test"]
print(f"treino={len(conjunto_treino)} | dev={len(conjunto_dev)} | teste={len(conjunto_teste)}")

Carregando splits oficiais do MilkQA...


treino=2307 | dev=50 | teste=300


In [8]:
print(f"Montando pares de treino (todas as {len(conjunto_treino)} perguntas, "
      f"{N_NEGATIVOS_TREINO} negativos cada, {FRACAO_NEGATIVOS_DIFICEIS:.0%} deles 'dificeis')...")
X_treino, y_treino = montar_pares(conjunto_treino, emb_perguntas, emb_respostas,
                                   N_NEGATIVOS_TREINO, fracao_dificeis=FRACAO_NEGATIVOS_DIFICEIS)
print(f"{X_treino.shape[0]} pares ({int(y_treino.sum())} positivos, {int((1 - y_treino).sum())} negativos)")

Montando pares de treino (todas as 2307 perguntas, 8 negativos cada, 50% deles 'dificeis')...


20763 pares (2307 positivos, 18456 negativos)


In [9]:
print("\n===== Baseline sem treino (cosseno puro) =====")
acuracia1_base, mrr_base = avaliar_cosseno_puro(conjunto_teste, emb_perguntas, emb_respostas)
print(f"Accuracy@1 = {acuracia1_base:.4f} | MRR = {mrr_base:.4f}")


===== Baseline sem treino (cosseno puro) =====
Accuracy@1 = 0.2767 | MRR = 0.3924


In [10]:
print("\n===== Grid Search (selecao pelo MRR no dev) =====")
combinacoes = list(itertools.product(
    GRADE_HIPERPARAMETROS["ocultas"],
    GRADE_HIPERPARAMETROS["dropout"],
    GRADE_HIPERPARAMETROS["taxa_aprendizado"],
))
resultados = []
melhor_geral = {"mrr": -1.0, "modelo": None, "config": None}

t0 = time.time()
for ocultas, dropout, taxa_aprendizado in combinacoes:
    modelo, mrr_dev = treinar_um_modelo(X_treino, y_treino, conjunto_dev, emb_perguntas,
                                         emb_respostas, ocultas, dropout, taxa_aprendizado)
    acuracia1_dev, _ = avaliar_ranking(modelo, conjunto_dev, emb_perguntas, emb_respostas)
    resultados.append({"ocultas": ocultas, "dropout": dropout, "taxa_aprendizado": taxa_aprendizado,
                        "mrr_dev": mrr_dev, "acuracia1_dev": acuracia1_dev})
    print(f"  ocultas={ocultas:4d} dropout={dropout:.1f} lr={taxa_aprendizado:.0e} "
          f"-> dev MRR={mrr_dev:.4f} Acc@1={acuracia1_dev:.4f}")
    if mrr_dev > melhor_geral["mrr"]:
        melhor_geral = {"mrr": mrr_dev, "modelo": modelo, "config": (ocultas, dropout, taxa_aprendizado)}

print(f"\nTempo do grid search: {time.time() - t0:.0f}s")


===== Grid Search (selecao pelo MRR no dev) =====


  ocultas= 128 dropout=0.2 lr=1e-03 -> dev MRR=0.7422 Acc@1=0.6800


  ocultas= 128 dropout=0.2 lr=1e-04 -> dev MRR=0.7372 Acc@1=0.6600


  ocultas= 128 dropout=0.4 lr=1e-03 -> dev MRR=0.7141 Acc@1=0.6200


  ocultas= 128 dropout=0.4 lr=1e-04 -> dev MRR=0.7402 Acc@1=0.6600


  ocultas= 256 dropout=0.2 lr=1e-03 -> dev MRR=0.7425 Acc@1=0.6600


  ocultas= 256 dropout=0.2 lr=1e-04 -> dev MRR=0.7261 Acc@1=0.6400


  ocultas= 256 dropout=0.4 lr=1e-03 -> dev MRR=0.7378 Acc@1=0.6600


  ocultas= 256 dropout=0.4 lr=1e-04 -> dev MRR=0.7441 Acc@1=0.6600

Tempo do grid search: 308s


In [11]:
ocultas, dropout, taxa_aprendizado = melhor_geral["config"]
print(f"Melhor configuracao: ocultas={ocultas}, dropout={dropout}, lr={taxa_aprendizado} "
      f"(dev MRR={melhor_geral['mrr']:.4f})")

print("\n===== Avaliacao final no TESTE (300 perguntas, 50 candidatas cada) =====")
modelo_final = melhor_geral["modelo"]
acuracia1_teste, mrr_teste = avaliar_ranking(modelo_final, conjunto_teste, emb_perguntas, emb_respostas)
print(f"Accuracy@1 (teste, MLP treinado) = {acuracia1_teste:.4f}")
print(f"MRR (teste, MLP treinado)        = {mrr_teste:.4f}")
print(f"Accuracy@1 (teste, cosseno cru)  = {acuracia1_base:.4f}")
print(f"MRR (teste, cosseno cru)         = {mrr_base:.4f}")

Melhor configuracao: ocultas=256, dropout=0.4, lr=0.0001 (dev MRR=0.7441)

===== Avaliacao final no TESTE (300 perguntas, 50 candidatas cada) =====


Accuracy@1 (teste, MLP treinado) = 0.5733
MRR (teste, MLP treinado)        = 0.6770
Accuracy@1 (teste, cosseno cru)  = 0.2767
MRR (teste, cosseno cru)         = 0.3924


In [12]:
os.makedirs("checkpoints", exist_ok=True)
torch.save({
    "model_state": modelo_final.state_dict(),
    "ocultas": ocultas, "dropout": dropout, "taxa_aprendizado": taxa_aprendizado,
    "dim_entrada": X_treino.shape[1],
    "acuracia1_teste": acuracia1_teste, "mrr_teste": mrr_teste,
}, "checkpoints/best_model_notebook.pt")
pd.DataFrame(resultados).sort_values("mrr_dev", ascending=False).to_csv(
    "checkpoints/grid_search_results_notebook.csv", index=False)
print("Checkpoint e grid search salvos em checkpoints/")

Checkpoint e grid search salvos em checkpoints/


## Conclusão

O MLP treinado sobre os embeddings congelados do BERTimbau chega em
`Accuracy@1 = 0,570` e `MRR = 0,679` no teste — quase o dobro do cosseno cru
(`0,277`), mas ainda atrás das variantes que usam fine-tuning ou fundem sinal
lexical (ver os outros notebooks do projeto). Fica claro aqui que o "salto"
não vem só de ter um embedding melhor: o classificador aprende uma noção de
correspondência pergunta-resposta que a similaridade bruta não captura
sozinha.